# Exploratory data analysis 1

**INPUT**: Raw corpus

**OUTPUT**: Preprocessing decisions + Stratified split

| Step | Decision | Status | Comment |
|------|----------|--------|---------|
| Stratified split | In setup part | Done |  |
| Columns & Shape | Drop some | Decided | Date not valuable for this project, drop in preprocessing |
| Columns Subject & Message | Merge | Decided | Own decision, merge in preprocessing |
| Empty values | Remove | Decided | Insignificant amount, remove in preprocessing |
| Duplicates | Remove | Decided | Insignificant amount, remove in preprocessing |
| Label balance | No action | Done | Near 50-50 |
| Message length distribution | Remove outliers | Decided | Use the 1.5xIQR (John Tukey) rule, lower is minus, add one manually |
| Patterns | Replace | Decided | Replace them in preprocessing if they can be renormalized (they are exploded with spaces) |
| Repeated chars | Collapse them | Decided | Same char appearing > 3 times. Collapse them to unified 2 chars |
| Special chars | Keep some | Pending | Maybe ! ? % $ are meaningful keep them? Ask Consultant |
| Vocabulary size estimation | Between 16-32k | Decided | For LSTM start with low |


## Input & Setup

### imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from urllib.request import urlretrieve
import zipfile
from sklearn.model_selection import train_test_split
import re
from dataclasses import fields
import sys
from pathlib import Path

project_root = Path().resolve().parent
src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from config.config import PATHS, REGEXES

### Build data directory tree

In [ ]:
PATHS.make_dirs()

### Download raw corpus and unzip it

In [ ]:
url = "https://github.com/MWiechmann/enron_spam_data/raw/master/enron_spam_data.zip"
zip_path = PATHS.corpora_raw / "enron_spam_data.zip"

urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(PATHS.corpora_raw)

### Read data

In [ ]:
corpus = pd.read_csv(PATHS.corpus_raw, index_col=0)

### Stratified split, and save

In [ ]:
train, temporary = train_test_split(
    corpus,
    test_size=0.4,
    random_state=42,
    stratify=corpus['Spam/Ham']
)
validation, test = train_test_split(
    temporary,
    test_size=0.5,
    random_state=42,
    stratify=temporary['Spam/Ham']
)
train.to_csv(PATHS.train_raw)
validation.to_csv(PATHS.validation_raw)
test.to_csv(PATHS.test_raw)

## Steps

### Columns and shape

In [ ]:
shape = corpus.shape
print(f"Number of entires: {shape[0]},\nNumber of columns: {shape[1]}")
print(f'Column names: {", ".join(corpus.columns)}')

### Detect empty values

In [ ]:
empty_values = corpus.isnull().sum()
print("Number of empty values:")
for col, count in empty_values.sort_values(ascending=False).items():
    print(f"{col:<15} {count:>5}")

### Detect duplicates

In [ ]:
count_duplicates = corpus.duplicated().sum()
print(f"Duplicates found: {count_duplicates}")

### Label balance inspection

In [ ]:
def class_distribution(name, corpus, label_column):
    num_of_labels = corpus[label_column].value_counts().to_dict()
    percentage_of_labels = {key:round(value/len(corpus)*100, 2) for key,value in num_of_labels.items()}
    print(f"Class distribution for {label_column} in {name}:")
    print(f"Number of labels:")
    for label, count in num_of_labels.items():
        print(f"{label.capitalize()}: {count}")
    
    print(f"Percentage of labels:")
    for label, percentage in percentage_of_labels.items():
        print(f"{label.capitalize()}: {percentage} %")
    print("\n")

class_distribution("corpus", corpus, "Spam/Ham")

### Patterns to replace

In [ ]:
def remove_spaces_around_spec_chars(text):
    text = str(text)
    text = re.sub(r"[ \t\xa0]*([:/?#[\]@!$&'()*+,;=%_.~-])[ \t\xa0]*", r"\1", text)
    return text

def number_of_patterns(column, regexes):
    for field in fields(regexes):
        k = field.name
        v = getattr(regexes, k)
        num = column.str.count(v).sum()
        print(f"{'Number of ' + k + 's:':<25} {num:>8g}")

corpus["Message"] = corpus["Message"].apply(remove_spaces_around_spec_chars)
number_of_patterns(corpus["Message"], REGEXES)

### Message length distribution

In [ ]:
msg_lengths = corpus["Message"].str.len()
char_length_statistics = msg_lengths.describe()

spam_msg_lengths = corpus[corpus['Spam/Ham'] == 'spam']['Message'].str.len()
spam_char_lengths_statistics = spam_msg_lengths.describe()

ham_msg_lengths = corpus[corpus['Spam/Ham'] == 'ham']['Message'].str.len()    
ham_char_lengths_statistics = ham_msg_lengths.describe()


def print_length_statistics(column):
    statistics = column.str.len().describe()
    for stat, value in statistics.items():
        print(f"{stat+':':<10} {value:>8.2f}")

print_length_statistics(corpus["Message"])

In [ ]:
plt.figure()
plt.hist(ham_msg_lengths, bins=50)
plt.hist(spam_msg_lengths, bins=50)
plt.xlabel("Message length (characters)")
plt.ylabel("Count")
plt.title("Message Length Distribution")
plt.show()

In [ ]:
q1 = char_length_statistics['25%']
q3 = char_length_statistics['75%']

def calculate_scaled_IQR(q1, q3, scaling_factor = 1.5):
    IQR = q3 -q1
    upper_boundary = int(q3 + scaling_factor*IQR)
    lower_boundary = int(q1 - scaling_factor*IQR)
    return upper_boundary, lower_boundary

upper, lower = calculate_scaled_IQR(q1,q3)
print("IQR rule outlier limits")
print(f"Upper limit: {upper}")
print(f"Lower limit: {lower}")
    

### Vocabulary size estimation

In [ ]:
messages = train['Message'].dropna().str.lower()
tokens = messages.apply(lambda x: re.findall(r"[a-z0-9]+", x))
vocab = {token for msg in tokens for token in msg}
vocab_size = len(vocab)

print("Estimated vocabulary size:", vocab_size)